# Route Optimisation Map — Streamlit Deployment

This notebook deploys a Streamlit in Snowflake app that visualises optimised delivery routes on an interactive PyDeck map.

The app calls the `TOOL_OPTIMIZATION` stored procedure (VROOM solver via OpenRouteService) and renders multi-vehicle routes with colour-coded paths.

**Prerequisites:**
- `OPENROUTESERVICE_APP` database exists with ORS services running
- `TOOL_OPTIMIZATION` stored procedure deployed
- Cortex Code skill `$route-optimisation-streamlit` available

## Step 1: Setup Session

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.query_tag = {"origin": "sf_sit-is", "name": "route_opt_streamlit", "version": {"major": 1, "minor": 0}}

## Step 2: Create Stage and File Format

In [ ]:
USE DATABASE OPENROUTESERVICE_APP;
USE SCHEMA CORE;

CREATE STAGE IF NOT EXISTS ROUTE_OPT_STREAMLIT_STAGE
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

CREATE OR REPLACE FILE FORMAT RAW_TEXT_FF
  TYPE = CSV COMPRESSION = NONE FIELD_DELIMITER = NONE
  RECORD_DELIMITER = NONE ESCAPE = NONE ESCAPE_UNENCLOSED_FIELD = NONE;

## Step 3: Upload environment.yml

In [ ]:
COPY INTO @OPENROUTESERVICE_APP.CORE.ROUTE_OPT_STREAMLIT_STAGE/environment.yml
FROM (SELECT $$name: streamlit_environment
channels:
  - snowflake
dependencies:
  - streamlit
  - snowflake-snowpark-python
  - pandas
  - pydeck
  - polyline
$$)
FILE_FORMAT = 'OPENROUTESERVICE_APP.CORE.RAW_TEXT_FF'
SINGLE = TRUE OVERWRITE = TRUE HEADER = FALSE;

## Step 4: Upload Streamlit App

The app code is defined by the `$route-optimisation-streamlit` Cortex Code skill. It renders route optimisation results on a PyDeck map with multi-vehicle colour-coded paths.

In [ ]:
import tempfile, os

app_code = '''import streamlit as st
import json
import pandas as pd
import pydeck as pdk

st.set_page_config(page_title="Route Optimisation Map", page_icon="\U0001f5fa\ufe0f", layout="wide")

CUSTOM_CSS = '<style>@import url("https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap");html, body, [class*="css"] { font-family: "Inter", -apple-system, BlinkMacSystemFont, sans-serif; }h1, h2, h3 { color: #11181C; }.metric-card { background: #F1F8FF; border: 1px solid #E6E8EB; border-radius: 12px; padding: 16px; text-align: center; }.metric-card .label { font-size: 13px; color: #687076; margin-bottom: 4px; }.metric-card .value { font-size: 24px; font-weight: 700; color: #11181C; }.metric-card .value.blue { color: #29B5E8; }.metric-card .value.green { color: #10B981; }</style>'
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

VEHICLE_COLORS = [[41, 181, 232, 200], [255, 159, 54, 200], [16, 185, 129, 200], [139, 92, 246, 200], [239, 68, 68, 200], [251, 191, 36, 200]]

DB = "OPENROUTESERVICE_APP"
SCHEMA = "CORE"

session = st.connection("snowflake").session()

SF_LOCATIONS = {"Union Square": [-122.4075, 37.7879], "Ferry Building": [-122.3936, 37.7955], "Fishermans Wharf": [-122.4177, 37.8080], "Golden Gate Park": [-122.4862, 37.7694], "Castro": [-122.4350, 37.7609], "Mission District": [-122.4194, 37.7599], "SOMA": [-122.3994, 37.7785], "North Beach": [-122.4064, 37.8005], "Pacific Heights": [-122.4350, 37.7925], "Haight-Ashbury": [-122.4477, 37.7692], "Nob Hill": [-122.4167, 37.7930], "Chinatown": [-122.4064, 37.7941], "Marina": [-122.4367, 37.8020], "Potrero Hill": [-122.4003, 37.7562], "Sunset District": [-122.4951, 37.7533], "Richmond District": [-122.4784, 37.7793], "Hayes Valley": [-122.4243, 37.7759], "Dogpatch": [-122.3907, 37.7618], "Embarcadero": [-122.3880, 37.7913], "Presidio": [-122.4662, 37.7989]}

DEPOT = [-122.4194, 37.7749]

st.title("Route Optimisation Map")
st.markdown("Configure delivery jobs and vehicles, then solve the Vehicle Routing Problem using VROOM.")

col_config, col_map = st.columns([1, 2])

with col_config:
    st.markdown("### Configuration")
    profile = st.selectbox("Travel Profile", ["cycling-electric", "driving-car", "driving-hgv"])
    num_vehicles = st.slider("Number of Vehicles", 1, 6, 3)
    selected_jobs = st.multiselect("Select Delivery Locations", list(SF_LOCATIONS.keys()), default=list(SF_LOCATIONS.keys())[:10])
    if st.button("Optimise Routes", type="primary", use_container_width=True):
        if len(selected_jobs) < 2:
            st.error("Select at least 2 delivery locations.")
        else:
            jobs_array = []
            for i, name in enumerate(selected_jobs):
                coord = SF_LOCATIONS[name]
                jobs_array.append({"id": i + 1, "location": coord, "service": 300})
            vehicles_array = []
            for v in range(num_vehicles):
                vehicles_array.append({"id": v + 1, "start": DEPOT, "end": DEPOT, "profile": profile})
            jobs_json = json.dumps(jobs_array)
            vehicles_json = json.dumps(vehicles_array)
            with st.spinner("Solving VRP..."):
                try:
                    result = session.sql(f"CALL {DB}.{SCHEMA}.TOOL_OPTIMIZATION(PARSE_JSON('{jobs_json}'), PARSE_JSON('{vehicles_json}'), '{profile}')").collect()
                    response_text = result[0][0] if result else None
                    if response_text:
                        st.session_state["opt_result"] = json.loads(response_text) if isinstance(response_text, str) else response_text
                        st.session_state["opt_jobs"] = selected_jobs
                        st.success("Optimisation complete!")
                    else:
                        st.error("Empty response from optimiser.")
                except Exception as e:
                    st.error(f"Error: {str(e)}")
    if "opt_result" in st.session_state:
        result = st.session_state["opt_result"]
        routes = result.get("routes", [])
        unassigned = result.get("unassigned", [])
        summary = result.get("summary", {})
        st.markdown("### Summary")
        mc1, mc2, mc3 = st.columns(3)
        with mc1:
            total_dist = summary.get("distance", 0)
            st.markdown(f'<div class="metric-card"><div class="label">Total Distance</div><div class="value blue">{total_dist/1000:.1f} km</div></div>', unsafe_allow_html=True)
        with mc2:
            total_dur = summary.get("duration", 0)
            st.markdown(f'<div class="metric-card"><div class="label">Total Duration</div><div class="value green">{total_dur//60} min</div></div>', unsafe_allow_html=True)
        with mc3:
            st.markdown(f'<div class="metric-card"><div class="label">Routes / Unassigned</div><div class="value">{len(routes)} / {len(unassigned)}</div></div>', unsafe_allow_html=True)
        st.markdown("### Route Details")
        for i, route in enumerate(routes):
            color_hex = ["#29B5E8", "#FF9F36", "#10B981", "#8B5CF6", "#EF4444", "#FBBF24"][i % 6]
            steps = route.get("steps", [])
            jobs_in_route = [s for s in steps if s.get("type") == "job"]
            dist_km = route.get("distance", 0) / 1000
            dur_min = route.get("duration", 0) // 60
            st.markdown(f'<span style="color:{color_hex};font-weight:700;">Vehicle {route.get("vehicle", i+1)}</span> \u2014 {len(jobs_in_route)} stops, {dist_km:.1f} km, {dur_min} min', unsafe_allow_html=True)

with col_map:
    layers = []
    view_state = pdk.ViewState(latitude=37.7749, longitude=-122.4194, zoom=12, pitch=0)
    if "opt_result" in st.session_state:
        result = st.session_state["opt_result"]
        routes = result.get("routes", [])
        for i, route in enumerate(routes):
            color = VEHICLE_COLORS[i % len(VEHICLE_COLORS)]
            geometry = route.get("geometry", "")
            if geometry:
                import polyline as pl
                coords = pl.decode(geometry)
                path_data = [{"path": [[lng, lat] for lat, lng in coords], "color": color}]
                layers.append(pdk.Layer("PathLayer", data=path_data, get_path="path", get_color="color", width_min_pixels=4, get_width=6))
            steps = route.get("steps", [])
            stop_data = []
            for step in steps:
                if step.get("type") == "job" and "location" in step:
                    loc = step["location"]
                    stop_data.append({"position": loc, "color": color, "id": step.get("id", "")})
            if stop_data:
                layers.append(pdk.Layer("ScatterplotLayer", data=stop_data, get_position="position", get_fill_color="color", get_radius=80, pickable=True))
        layers.append(pdk.Layer("ScatterplotLayer", data=[{"position": DEPOT, "color": [0, 0, 0, 255]}], get_position="position", get_fill_color="color", get_radius=120))
    else:
        marker_data = [{"position": coord, "name": name} for name, coord in list(SF_LOCATIONS.items())[:10]]
        layers.append(pdk.Layer("ScatterplotLayer", data=marker_data, get_position="position", get_fill_color=[41, 181, 232, 200], get_radius=60, pickable=True))
        layers.append(pdk.Layer("ScatterplotLayer", data=[{"position": DEPOT}], get_position="position", get_fill_color=[0, 0, 0, 255], get_radius=120))
    deck = pdk.Deck(layers=layers, initial_view_state=view_state, map_style="mapbox://styles/mapbox/light-v11")
    st.pydeck_chart(deck, use_container_width=True, height=700)
'''

session.sql("REMOVE @OPENROUTESERVICE_APP.CORE.ROUTE_OPT_STREAMLIT_STAGE/streamlit_app.py").collect()

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(app_code)
    tmp_path = f.name

session.file.put(tmp_path, '@OPENROUTESERVICE_APP.CORE.ROUTE_OPT_STREAMLIT_STAGE', auto_compress=False, overwrite=True)
os.unlink(tmp_path)
print('streamlit_app.py uploaded successfully')


## Step 5: Verify Uploaded Files

In [ ]:
LS @OPENROUTESERVICE_APP.CORE.ROUTE_OPT_STREAMLIT_STAGE;

## Step 6: Create Streamlit App

In [ ]:
CREATE OR REPLACE STREAMLIT OPENROUTESERVICE_APP.CORE.ROUTE_OPTIMISATION_MAP
  FROM @OPENROUTESERVICE_APP.CORE.ROUTE_OPT_STREAMLIT_STAGE
  MAIN_FILE = 'streamlit_app.py'
  QUERY_WAREHOUSE = 'ROUTING_WH'
  TITLE = 'Route Optimisation Map';

In [ ]:
ALTER STREAMLIT OPENROUTESERVICE_APP.CORE.ROUTE_OPTIMISATION_MAP ADD LIVE VERSION FROM LAST;

## Step 7: Verify Deployment

In [ ]:
SHOW STREAMLITS LIKE 'ROUTE_OPTIMISATION_MAP' IN SCHEMA OPENROUTESERVICE_APP.CORE;

## Done!

Your Route Optimisation Map Streamlit app is now deployed. Open it from Snowsight under **Streamlit Apps** > **ROUTE_OPTIMISATION_MAP**.

The app allows you to:
- Select delivery locations across San Francisco
- Choose travel profile (e-bike, car, HGV)
- Set number of vehicles (1-6)
- Solve the VRP and see colour-coded routes on the map
- View per-vehicle metrics (distance, duration, stops)

**Cortex Code Skill:** This app was built using the `$route-optimisation-streamlit` skill.